In [ ]:
import time,datetime,json,urllib.request,subprocess,sys
start=time.monotonic()
print(datetime.datetime.now(datetime.timezone.utc).isoformat(),"START final report supplement",flush=True)
try:
    subprocess.run([sys.executable,"-m","pip","install","-q","huggingface_hub==1.33.0"],check=True)
    from huggingface_hub import HfApi
    from kaggle_secrets import UserSecretsClient
    secrets=UserSecretsClient()
    try:key=secrets.get_secret("HF_WRITE_TOKEN")
    except Exception:key=secrets.get_secret("HF_WRITE")
    api=HfApi(token=key)
    base="https://raw.githubusercontent.com/dixitkeshav/finsafe/b4bf0f816d28120c8661efffae9891e0c01f1fc2/"
    def read(url):
        with urllib.request.urlopen(url,timeout=60) as response:return response.read()
    article=read(base+"README.md").decode().replace("](docs/","](https://github.com/dixitkeshav/finsafe/blob/main/docs/")
    api.upload_file(path_or_fileobj=article.encode(),path_in_repo="ARTICLE.md",repo_id="Finsafe-team/finsafe-runs",repo_type="dataset")
    for source,target in [("data/reviewed-validation-v1/sensitivity.json","review-finalization/sensitivity.json"),("docs/REPRODUCED.json","public-reproduction/completed.json")]:
        api.upload_file(path_or_fileobj=read(base+source),path_in_repo=target,repo_id="Finsafe-team/finsafe-runs",repo_type="dataset")
    page=read("https://huggingface.co/spaces/Finsafe-team/finsafe/resolve/main/index.html").decode()
    marker='<div class="scroll">'
    if 'Structured finance output:' not in page:
        page=page.replace(marker,'<p><strong>Structured finance output:</strong> valid JSON improved from 64% aligned to 94% after SFT. Correct numerical answers remain low (7% after SFT), so this is a research result, not a production financial adviser. Zero observed attacks from twelve prompts does not establish zero risk.</p>'+marker)
    api.upload_file(path_or_fileobj=page.encode(),path_in_repo="index.html",repo_id="Finsafe-team/finsafe",repo_type="space")
    print("Updated full report, label sensitivity, anonymous reproduction evidence and static demo",flush=True)
except BaseException as error:
    print("FAILED supplement",type(error).__name__,round(time.monotonic()-start,1),flush=True)
    raise
else:print("DONE supplement",round(time.monotonic()-start,1),"seconds",flush=True)
